In [13]:
import duckdb
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

RAW = Path('../data/raw')


hospitals = {
    'baylor':                RAW / 'baylor_university_medical_center-69947_parsed.duckdb',
    'methodist':             RAW / 'methodist_dallas_medical_center-6000b_parsed.duckdb',
    'parkland':              RAW / 'parkland_health-6e88d_parsed.duckdb',
    'texas_health_plano':    RAW / 'texas_health_presbyterian_hospital_plano-6ad81_parsed.duckdb',
    'medical_city_alliance': RAW / 'medical_city_alliance_hospital-77912_parsed.duckdb',
}

for name, path in hospitals.items():
    print(f"{name:25s} {'OK' if path.exists() else 'MISSING'}  {path.name}")

baylor                    OK  baylor_university_medical_center-69947_parsed.duckdb
methodist                 OK  methodist_dallas_medical_center-6000b_parsed.duckdb
parkland                  OK  parkland_health-6e88d_parsed.duckdb
texas_health_plano        OK  texas_health_presbyterian_hospital_plano-6ad81_parsed.duckdb
medical_city_alliance     OK  medical_city_alliance_hospital-77912_parsed.duckdb


In [9]:
RAW = Path('../data/raw')
for f in sorted(RAW.iterdir()):
    print(f.name)

.gitkeep
baylor_university_medical_center-69947_parsed.duckdb
medical_city_alliance_hospital-77912_parsed.duckdb
methodist_dallas_medical_center-6000b_parsed.duckdb
parkland_health-6e88d_parsed.duckdb
texas_health_presbyterian_hospital_plano-6ad81_parsed.duckdb


In [7]:
import os
print("CWD:", os.getcwd())
print()
print("Contents:")
for item in sorted(os.listdir('.')):
    print(" ", item)

CWD: c:\Users\kedha\Documents\dfw-hospital-pricing\notebooks

Contents:
  01_sanity_check.ipynb
  02_first_pull.ipynb
  3_anomaly_investigation.ipynb


In [12]:
print("RAW resolves to:", RAW.resolve())
print("RAW exists:", RAW.exists())
print()

# Compare expected names vs actual names byte-for-byte
actual_names = {f.name for f in RAW.iterdir()} if RAW.exists() else set()
for name, path in hospitals.items():
    expected = path.name
    match = expected in actual_names
    print(f"{name:25s} {'OK' if match else 'MISMATCH'}  expected: {expected!r}")

print()
print("Actually in folder:")
for n in sorted(actual_names):
    print(f"  {n!r}")

RAW resolves to: C:\Users\kedha\Documents\dfw-hospital-pricing\notebooks\data\raw
RAW exists: False

baylor                    MISMATCH  expected: 'baylor_university_medical_center-69947_parsed.duckdb'
methodist                 MISMATCH  expected: 'methodist_dallas_medical_center-6000b_parsed.duckdb'
parkland                  MISMATCH  expected: 'parkland_health-6e88d_parsed.duckdb'
texas_health_plano        MISMATCH  expected: 'texas_health_presbyterian_hospital_plano-6ad81_parsed.duckdb'
medical_city_alliance     MISMATCH  expected: 'medical_city_alliance_hospital-77912_parsed.duckdb'

Actually in folder:


In [14]:
con = duckdb.connect(str(hospitals['methodist']), read_only=True)


tables = con.execute("SHOW TABLES").fetchdf()
print("Tables:")
print(tables)
print()


for t in tables['name']:
    print(f"--- {t} ---")
    cols = con.execute(f"DESCRIBE {t}").fetchdf()
    print(cols[['column_name', 'column_type']].to_string(index=False))
    print()

Tables:
                      name
0         cms_hpt_metadata
1                hospitals
2  modifier_charge_details
3         modifier_charges
4             mrf_metadata
5  standard_charge_details
6         standard_charges

--- cms_hpt_metadata ---
    column_name column_type
  contact_email     VARCHAR
   contact_name     VARCHAR
  location_name     VARCHAR
        mrf_url     VARCHAR
source_page_url     VARCHAR

--- hospitals ---
                column_name column_type
                hospital_id      BIGINT
              hospital_name     VARCHAR
           hospital_address     VARCHAR
              location_name     VARCHAR
            last_updated_on        DATE
                    version     VARCHAR
             license_number     VARCHAR
             hospital_state     VARCHAR
                attestation     VARCHAR
        confirm_attestation     BOOLEAN
              attester_name     VARCHAR
                 type_2_npi   VARCHAR[]
       financial_aid_policy     VARCHAR
gen

In [15]:
pd.set_option('display.max_rows', None)

for t in ['standard_charges', 'standard_charge_details', 
         'modifier_charges', 'modifier_charge_details']:
    print(f"=========== {t} ===========")
    cols = con.execute(f"DESCRIBE {t}").fetchdf()
    print(cols[['column_name', 'column_type']].to_string(index=False))
    print()


print("=========== row counts ===========")
for t in ['standard_charges', 'standard_charge_details',
         'modifier_charges', 'modifier_charge_details']:
    n = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t:30s} {n:>12,}")

=========== standard_charges ===========
             column_name column_type
               charge_id      BIGINT
              charge_seq     INTEGER
             hospital_id      BIGINT
             description     VARCHAR
            gross_charge      DOUBLE
         discounted_cash      DOUBLE
                 minimum      DOUBLE
                 maximum      DOUBLE
                 setting     VARCHAR
           billing_class     VARCHAR
               modifiers     VARCHAR
additional_generic_notes     VARCHAR
               drug_unit     VARCHAR
               drug_type     VARCHAR
                     cpt     VARCHAR
                   hcpcs     VARCHAR
                  ms_drg     VARCHAR
                      rc     VARCHAR
                     cdm     VARCHAR
                     ndc     VARCHAR
                     icd     VARCHAR
             other_code1     VARCHAR
        other_code1_type     VARCHAR
             other_code2     VARCHAR
        other_code2_type     VARCH

In [16]:
print("=== standard_charge_details (FULL) ===")
print(con.execute("DESCRIBE standard_charge_details").fetchdf().to_string(index=False))
print()
print("=== standard_charges (FULL) ===")
print(con.execute("DESCRIBE standard_charges").fetchdf().to_string(index=False))

=== standard_charge_details (FULL) ===
               column_name column_type null  key default extra
                 detail_id      BIGINT   NO None    None  None
                 charge_id      BIGINT  YES None    None  None
                charge_seq     INTEGER  YES None    None  None
                 payer_seq     INTEGER  YES None    None  None
               hospital_id      BIGINT  YES None    None  None
               description     VARCHAR  YES None    None  None
              gross_charge      DOUBLE  YES None    None  None
           discounted_cash      DOUBLE  YES None    None  None
                   minimum      DOUBLE  YES None    None  None
                   maximum      DOUBLE  YES None    None  None
                   setting     VARCHAR  YES None    None  None
             billing_class     VARCHAR  YES None    None  None
  additional_generic_notes     VARCHAR  YES None    None  None
                 drug_unit     VARCHAR  YES None    None  None
                

In [17]:
with pd.option_context('display.max_rows', None):
    print("=== standard_charge_details (FULL) ===")
    print(con.execute("DESCRIBE standard_charge_details").fetchdf()[['column_name', 'column_type']].to_string(index=False))

=== standard_charge_details (FULL) ===
               column_name column_type
                 detail_id      BIGINT
                 charge_id      BIGINT
                charge_seq     INTEGER
                 payer_seq     INTEGER
               hospital_id      BIGINT
               description     VARCHAR
              gross_charge      DOUBLE
           discounted_cash      DOUBLE
                   minimum      DOUBLE
                   maximum      DOUBLE
                   setting     VARCHAR
             billing_class     VARCHAR
  additional_generic_notes     VARCHAR
                 drug_unit     VARCHAR
                 drug_type     VARCHAR
                       cpt     VARCHAR
                     hcpcs     VARCHAR
                    ms_drg     VARCHAR
                        rc     VARCHAR
                       cdm     VARCHAR
                       ndc     VARCHAR
                       icd     VARCHAR
               other_code1     VARCHAR
          other_code1_typ

In [18]:
cols = con.execute("DESCRIBE standard_charge_details").fetchdf()
print(f"Total columns: {len(cols)}")
print()
for i, row in cols.iterrows():
    print(f"  {i:2d}. {row['column_name']:35s} {row['column_type']}")

Total columns: 44

   0. detail_id                           BIGINT
   1. charge_id                           BIGINT
   2. charge_seq                          INTEGER
   3. payer_seq                           INTEGER
   4. hospital_id                         BIGINT
   5. description                         VARCHAR
   6. gross_charge                        DOUBLE
   7. discounted_cash                     DOUBLE
   8. minimum                             DOUBLE
   9. maximum                             DOUBLE
  10. setting                             VARCHAR
  11. billing_class                       VARCHAR
  12. additional_generic_notes            VARCHAR
  13. drug_unit                           VARCHAR
  14. drug_type                           VARCHAR
  15. cpt                                 VARCHAR
  16. hcpcs                               VARCHAR
  17. ms_drg                              VARCHAR
  18. rc                                  VARCHAR
  19. cdm                             

In [19]:
cols = con.execute("DESCRIBE standard_charge_details").fetchdf()
out = "\n".join(f"{i:2d}. {r['column_name']:40s} {r['column_type']}" 
                for i, r in cols.iterrows())
Path('../scratch_schema.txt').write_text(out)
print(f"Wrote {len(cols)} columns to scratch_schema.txt")

Wrote 44 columns to scratch_schema.txt


In [21]:
q = """
SELECT 
    scd.detail_id,
    sc.cpt, sc.hcpcs,
    sc.setting, sc.billing_class, sc.modifiers,
    sc.gross_charge,
    scd.payer_name, scd.plan_name, scd.payer_group, scd.payer_type,
    scd.methodology, scd.methodology_normalized,
    scd.standard_charge_dollar,
    scd.standard_charge_percentage,
    scd.standard_charge_algorithm,
    scd.estimated_amount,
    scd.additional_payer_notes
FROM standard_charges sc
JOIN standard_charge_details scd ON sc.charge_id = scd.charge_id
WHERE (sc.cpt = '73721' OR sc.hcpcs = '73721')
  AND scd.payer_name ILIKE '%united%'
ORDER BY scd.standard_charge_dollar, scd.plan_name
"""

methodist_uhc = con.execute(q).fetchdf()
print(f"Rows returned: {len(methodist_uhc)}")
methodist_uhc

Rows returned: 6


,detail_id,cpt,hcpcs,setting,billing_class,modifiers,gross_charge,payer_name,plan_name,payer_group,payer_type,methodology,methodology_normalized,standard_charge_dollar,standard_charge_percentage,standard_charge_algorithm,estimated_amount,additional_payer_notes
0,2329334,73721,None,outpatient,None,None,4878.0,UNITED HEALTHCARE MEDICAID MANAGED CARE [5015],MHS HB UNITED MEDICAID STAR PLUS MDMC,UnitedHealthcare,Commercial,fee schedule,fee schedule,200.30,NaN,NaN,NaN,Radiology; 95% of FSC: MHS HB XR MEDICAID MHS ...
1,2329333,73721,None,outpatient,None,None,4878.0,UNITED HEALTHCARE MANAGED CARE [3021],MHS HB UHC EXCHANGE MDMC,UnitedHealthcare,Commercial,fee schedule,fee schedule,232.47,NaN,Includes reimbursement for any outliers.,NaN,OPPS APC; APC Pricing (APC Code: 5523 / Adjus...
2,2329335,73721,None,outpatient,None,None,4878.0,UNITED HEALTHCARE MEDICARE MANAGED CARE [3044],MHS HB UHC MEDICARE COMPLETE MDMC,UnitedHealthcare,Commercial,fee schedule,fee schedule,232.47,NaN,Includes reimbursement for any outliers.,NaN,OPPS APC; APC Pricing (APC Code: 5523 / Adjus...
3,2329336,73721,None,outpatient,None,None,4878.0,UNITED HEALTHCARE MEDICARE MANAGED CARE [7010],MHS HB UHC MEDICARE COMPLETE MDMC,UnitedHealthcare,Commercial,fee schedule,fee schedule,232.47,NaN,Includes reimbursement for any outliers.,NaN,OPPS APC; APC Pricing (APC Code: 5523 / Adjus...
4,2329337,73721,None,outpatient,None,None,4878.0,UNITED HEALTHCARE MEDICARE MANAGED CARE [7010],MHS HB UHC MEDICARE DIRECT MDMC,UnitedHealthcare,Commercial,fee schedule,fee schedule,232.47,NaN,Includes reimbursement for any outliers.,NaN,OPPS APC; APC Pricing (APC Code: 5523 / Adjus...
5,2329338,73721,None,outpatient,None,None,4878.0,UNITED HEALTHCARE MEDICARE MANAGED CARE [7010],MHS HB UHC WELLMED MDMC,UnitedHealthcare,Commercial,fee schedule,fee schedule,232.47,NaN,Includes reimbursement for any outliers.,NaN,OPPS APC; APC Pricing (APC Code: 5523 / Adjus...


In [22]:
methodist_uhc[[
    'plan_name', 'payer_group', 'payer_type', 
    'methodology', 'methodology_normalized',
    'standard_charge_dollar', 'standard_charge_percentage', 
    'standard_charge_algorithm', 'estimated_amount',
    'setting', 'billing_class'
]]

,plan_name,payer_group,payer_type,methodology,methodology_normalized,standard_charge_dollar,standard_charge_percentage,standard_charge_algorithm,estimated_amount,setting,billing_class
0,MHS HB UNITED MEDICAID STAR PLUS MDMC,UnitedHealthcare,Commercial,fee schedule,fee schedule,200.30,NaN,NaN,NaN,outpatient,None
1,MHS HB UHC EXCHANGE MDMC,UnitedHealthcare,Commercial,fee schedule,fee schedule,232.47,NaN,Includes reimbursement for any outliers.,NaN,outpatient,None
2,MHS HB UHC MEDICARE COMPLETE MDMC,UnitedHealthcare,Commercial,fee schedule,fee schedule,232.47,NaN,Includes reimbursement for any outliers.,NaN,outpatient,None
3,MHS HB UHC MEDICARE COMPLETE MDMC,UnitedHealthcare,Commercial,fee schedule,fee schedule,232.47,NaN,Includes reimbursement for any outliers.,NaN,outpatient,None
4,MHS HB UHC MEDICARE DIRECT MDMC,UnitedHealthcare,Commercial,fee schedule,fee schedule,232.47,NaN,Includes reimbursement for any outliers.,NaN,outpatient,None
5,MHS HB UHC WELLMED MDMC,UnitedHealthcare,Commercial,fee schedule,fee schedule,232.47,NaN,Includes reimbursement for any outliers.,NaN,outpatient,None


In [23]:
q = """
SELECT DISTINCT
    scd.payer_name, scd.plan_name, scd.payer_group, scd.payer_type,
    scd.methodology_normalized,
    scd.standard_charge_dollar
FROM standard_charges sc
JOIN standard_charge_details scd ON sc.charge_id = scd.charge_id
WHERE (sc.cpt = '73721' OR sc.hcpcs = '73721')
  AND sc.setting = 'outpatient'
  AND scd.standard_charge_dollar IS NOT NULL
ORDER BY scd.payer_group, scd.standard_charge_dollar
"""
methodist_all = con.execute(q).fetchdf()
print(f"Total Methodist 73721 outpatient rate rows: {len(methodist_all)}")
methodist_all

Total Methodist 73721 outpatient rate rows: 82


,payer_name,plan_name,payer_group,payer_type,methodology_normalized,standard_charge_dollar
0,AETNA [3000],MHS HB AETNA EXCHANGE MDMC,Aetna,Commercial,fee schedule,232.47
1,AETNA [3000],MHS HB AETNA CVS MDMC,Aetna,Commercial,fee schedule,232.47
2,AETNA MEDICARE MANAGED CARE [7000],MHS HB AETNA GOLDEN MDMC,Aetna,Commercial,fee schedule,232.47
3,AETNA MANAGED CARE [2068],MHS HB AETNA TX PREF PLUS II MDMC,Aetna,Commercial,per diem,1518.00
4,AETNA COMMERCIAL [2042],MHS HB AETNA TX PREF PLUS II MDMC,Aetna,Commercial,per diem,1518.00
5,AETNA [3000],MHS HB AETNA BAYLOR SCOTT AND WHITE MDMC,Aetna,Commercial,per diem,1592.00
6,AETNA MANAGED CARE [2068],MHS HB AETNA TEXAS ADVANTAGE MDMC,Aetna,Commercial,per diem,1608.00
7,AETNA COMMERCIAL [2042],MHS HB AETNA TEXAS ADVANTAGE MDMC,Aetna,Commercial,per diem,1608.00
8,AETNA MANAGED CARE [2068],MHS HB AETNA WHOLE HEALTH MDMC,Aetna,Commercial,per diem,1608.00
9,AETNA [3000],MHS HB AETNA HMO PPO MDMC,Aetna,Commercial,per diem,1787.00


In [24]:

summary = methodist_all.groupby(['payer_group', 'methodology_normalized']).agg(
    n_plans=('plan_name', 'count'),
    min_rate=('standard_charge_dollar', 'min'),
    median_rate=('standard_charge_dollar', 'median'),
    max_rate=('standard_charge_dollar', 'max'),
).round(2).reset_index()
summary

,payer_group,methodology_normalized,n_plans,min_rate,median_rate,max_rate
0,Aetna,fee schedule,3,232.47,232.47,232.47
1,Aetna,per diem,8,1518.00,1608.00,1787.00
2,Aetna,percent of total billed charges,1,3414.60,3414.60,3414.60
3,BCBS,fee schedule,9,232.47,1381.93,1593.99
4,Cigna,fee schedule,3,232.47,232.47,232.47
5,Humana,fee schedule,3,232.47,232.47,232.47
6,Medicaid,fee schedule,2,862.76,969.34,1075.92
7,Medicare,fee schedule,9,232.47,232.47,232.47
8,Molina,fee schedule,3,232.47,232.47,862.76
9,Other,case rate,1,850.00,850.00,850.00


In [25]:

n_total = len(methodist_all)
n_232 = (methodist_all['standard_charge_dollar'] == 232.47).sum()
print(f"Total Methodist 73721 outpatient rate rows: {n_total}")
print(f"Rows at exactly $232.47:                   {n_232}  ({100*n_232/n_total:.0f}%)")

Total Methodist 73721 outpatient rate rows: 82
Rows at exactly $232.47:                   58  (71%)


In [ ]:
## Methodist UHC anomaly — resolved

**Day 3 claim:** Methodist UnitedHealthcare priced knee MRI (73721) at a flat 
$232.47 across 7 plans, suggesting either a flat-rate contract or a data 
quality issue.

**Day 4 finding:** Neither. The $232.47 is the **Medicare fee schedule allowable** 
for outpatient CPT 73721. It appears as the negotiated rate across every 
insurer's government-line products at Methodist — not just UHC.

### Evidence

- **58 of 82 outpatient 73721 rate rows (71%)** at Methodist Dallas are exactly $232.47.
- The pattern spans Aetna, BCBS, Cigna, Humana, Molina, UHC, and ~30 plans 
  Trilliant grouped as "Other" — every payer with Medicare Advantage, Medicaid 
  managed care, or ACA Exchange products contributes.
- The 7 UHC plans Day 3 flagged were a mix of Medicaid STAR PLUS, ACA Exchange, 
  and three Medicare Advantage products (Medicare Complete, Medicare Direct, 
  WellMed) — not commercial plans at all.

### What the actual commercial rates look like at Methodist

After filtering to genuinely commercial plans (per diem and commercial fee 
schedules), the outpatient knee MRI range is roughly **$836–$1,787**, with 
medians in the $1,400–$1,600 area. The Aetna Transplant Network plan (priced 
as % of total billed charges) resolves to **$3,414** — 70% of Methodist's 
$4,878 gross charge.

### Trilliant data quality flag

The `payer_type` column labels every UHC product as "Commercial," including 
STAR PLUS Medicaid and Medicare Advantage products. This is wrong by any 
reasonable definition. `payer_group` and `methodology_normalized` are more 
trustworthy; for line-of-business classification, plan name strings 
(STAR PLUS / MEDICARE / EXCHANGE) are the most reliable signal.

### Implication for Day 3 cross-hospital chart

The Day 3 chart almost certainly conflated government and commercial rates at 
every hospital. The "7x variation across DFW" headline is real but its 
composition is muddier than reported. **Action item (Day 5 candidate):** 
re-do the cross-hospital comparison filtered to commercial-only plans.

### Lessons

- Never trust a normalized categorical column without spot-checking it against 
  the underlying free-text fields (here: `plan_name` vs `payer_type`).
- "Flat-rate clusters" in hospital pricing data are usually the **Medicare 
  allowable** leaking through, not a real flat-rate contract.
- Filter by `methodology_normalized` before comparing rates across plans — 
  comparing a fee-schedule dollar to a percent-of-charge or per-diem rate is 
  apples to oranges.

In [26]:
con.close()
con = duckdb.connect(str(hospitals['parkland']), read_only=True)


tables = con.execute("SHOW TABLES").fetchdf()
print(tables['name'].tolist())

['cms_hpt_metadata', 'hospitals', 'modifier_charge_details', 'modifier_charges', 'mrf_metadata', 'standard_charge_details', 'standard_charges']


In [27]:
q = """
SELECT DISTINCT
    scd.payer_name, scd.plan_name, scd.payer_group, scd.payer_type,
    scd.methodology_normalized,
    scd.standard_charge_dollar,
    scd.standard_charge_percentage,
    scd.estimated_amount
FROM standard_charges sc
JOIN standard_charge_details scd ON sc.charge_id = scd.charge_id
WHERE (sc.cpt = '73721' OR sc.hcpcs = '73721')
  AND sc.setting = 'outpatient'
ORDER BY scd.payer_group, scd.standard_charge_dollar
"""
parkland_all = con.execute(q).fetchdf()
print(f"Total Parkland 73721 outpatient rate rows: {len(parkland_all)}")
parkland_all.head(20)

Total Parkland 73721 outpatient rate rows: 134


,payer_name,plan_name,payer_group,payer_type,methodology_normalized,standard_charge_dollar,standard_charge_percentage,estimated_amount
0,AETNA BETTER HEALTH [1317],ABOVE FPIL AETNA CHIP PERINATE [131703],Aetna,Commercial,fee schedule,208.80,NaN,NaN
1,AETNA BETTER HEALTH [1317],BELOW FPIL AETNA CHIP PERINATE [131702],Aetna,Commercial,fee schedule,208.80,NaN,NaN
2,AETNA BETTER HEALTH [1317],AETNA BETTER HEALTH CHIP [131701],Aetna,Commercial,fee schedule,208.80,NaN,NaN
3,AETNA BETTER HEALTH [1317],AETNA BETTER HEALTH STAR [131700],Aetna,Commercial,fee schedule,208.80,NaN,NaN
4,AETNA BETTER HEALTH [1317],ABOVE FPIL AETNA CHIP PERINATE [131703],Aetna,Commercial,percent of total billed charges,963.44,15.88,NaN
5,AETNA BETTER HEALTH [1317],BELOW FPIL AETNA CHIP PERINATE [131702],Aetna,Commercial,percent of total billed charges,963.44,15.88,NaN
6,AETNA BETTER HEALTH [1317],AETNA BETTER HEALTH STAR [131700],Aetna,Commercial,percent of total billed charges,963.44,15.88,NaN
7,AETNA BETTER HEALTH [1317],AETNA BETTER HEALTH CHIP [131701],Aetna,Commercial,percent of total billed charges,963.44,15.88,NaN
8,AETNA BETTER HEALTH [1317],AETNA BETTER HEALTH STAR [131700],Aetna,Commercial,percent of total billed charges,1926.40,15.88,NaN
9,AETNA BETTER HEALTH [1317],AETNA BETTER HEALTH CHIP [131701],Aetna,Commercial,percent of total billed charges,1926.40,15.88,NaN


In [28]:
gross = con.execute("""
    SELECT DISTINCT sc.setting, sc.billing_class, sc.gross_charge, sc.description
    FROM standard_charges sc
    WHERE (sc.cpt = '73721' OR sc.hcpcs = '73721')
""").fetchdf()
gross

,setting,billing_class,gross_charge,description
0,outpatient,facility,6067.0,MRI scan of leg joint without contrast
1,outpatient,facility,12131.0,MRI scan of leg joint without contrast


In [29]:
summary = parkland_all.groupby(['payer_group', 'methodology_normalized']).agg(
    n_plans=('plan_name', 'count'),
    min_rate=('standard_charge_dollar', 'min'),
    median_rate=('standard_charge_dollar', 'median'),
    max_rate=('standard_charge_dollar', 'max'),
    n_pct=('standard_charge_percentage', lambda x: x.notna().sum()),
).round(2).reset_index()
summary

,payer_group,methodology_normalized,n_plans,min_rate,median_rate,max_rate,n_pct
0,Aetna,fee schedule,4,208.80,208.80,208.80,0
1,Aetna,percent of total billed charges,10,963.44,1926.40,7278.60,10
2,BCBS,fee schedule,1,678.20,678.20,678.20,0
3,BCBS,percent of total billed charges,22,1456.08,4368.24,8734.32,22
4,Cigna,other,2,NaN,NaN,NaN,0
5,Cigna,percent of total billed charges,4,2851.50,4670.90,7278.60,4
6,Medicare,other,2,NaN,NaN,NaN,0
7,Medicare,percent of total billed charges,8,1268.00,1904.72,2547.51,8
8,Molina,fee schedule,4,229.70,542.15,886.70,0
9,Molina,other,9,NaN,NaN,NaN,0


In [30]:

n_total = len(parkland_all)
n_dollar = parkland_all['standard_charge_dollar'].notna().sum()
n_pct = parkland_all['standard_charge_percentage'].notna().sum()
print(f"Total rate rows:        {n_total}")
print(f"With dollar rate:       {n_dollar}  ({100*n_dollar/n_total:.0f}%)")
print(f"With percentage rate:   {n_pct}  ({100*n_pct/n_total:.0f}%)")

Total rate rows:        134
With dollar rate:       93  (69%)
With percentage rate:   72  (54%)


In [ ]:
## Parkland commercial-rate anomaly — resolved

**Day 3 claim:** Parkland (safety-net) had commercial rates HIGHER than 
the commercial systems (median BCBS rate $4,368), which was counter-intuitive.

**Day 4 finding:** Parkland and Methodist use fundamentally different pricing 
methodologies, and Day 3's cross-hospital comparison conflated them.

### Evidence

- **54% of Parkland's 73721 outpatient rate rows are "percent of total billed 
  charges"** — at Methodist, this was a fringe methodology (1 plan). At Parkland 
  it's the dominant structure for BCBS, Cigna, UHC, and Molina commercial plans.
- **Parkland has two outpatient `charge_id`s for 73721** with gross charges of 
  $6,067 and $12,131 (exactly 2x). Day 3 likely averaged percentage rates from 
  both, pulling the median upward.
- **The "$4,368 BCBS median" is a calculated value**, not a negotiated dollar 
  rate. It is 15.88%-ish of an inflated chargemaster price. No one likely pays 
  that exact figure in practice; the contract pays a percentage of whatever 
  the actual bill turns out to be.
- **Aetna Better Health at $208.80** is Texas CHIP (children's program), not 
  commercial — same `payer_type` mislabel as Methodist's UHC government products.

### Structural insight: why safety-net hospitals price this way

Percent-of-charges contracts with inflated chargemasters are a common safety-net 
hospital strategy. The inflated gross charge serves dual purposes: 
disproportionate-share (DSH) reimbursement calculations and as a base for any 
remaining percent-of-charge contracts. Major commercial insurers (BCBS, UHC, 
Cigna) appear to lack the leverage at Parkland to insist on flat negotiated 
rates the way they do at Methodist or Baylor.

### Implication for the Day 3 cross-hospital chart

The "7x variation across DFW" finding is real but conflates three different 
things being treated as comparable rates:
1. Medicare fee-schedule allowables (~$200–$232 — government-set, identical 
   across hospitals)
2. Genuine commercial negotiated dollar rates (Methodist/Baylor: ~$1,400–$1,800)
3. Calculated %-of-charges values on inflated chargemasters (Parkland: $1,400–$8,700+)

These cannot meaningfully be compared on the same axis.

### Lesson

For any future cross-hospital pricing comparison: filter to a single 
`methodology_normalized` value first, then compare. Mixing methodologies makes 
the headline number meaningless. The clean comparison is 
**commercial fee-schedule rates only**, hospital-by-hospital.